# Resident health trajectory — next score from lags


## 1. Business Understanding

**PREDICTIVE regression.** Forecast **next** `general_health_score` from prior observations to support clinical check-in scheduling.

**Success:** Test MAE < **0.25** on 1–5 scale (context-dependent).


## 2. Data Understanding & Preparation (EDA)


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
import subprocess, sys, warnings
warnings.filterwarnings("ignore")

def _find_ml_dir() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        b = p / "build_master_datasets.py"
        d = p / "data" / "supporters.csv"
        if b.exists() and d.exists():
            return p
        v2 = p / "ml-pipelines-v2"
        if (v2 / "build_master_datasets.py").exists():
            return v2
        p = p.parent
    raise FileNotFoundError("Could not find ml-pipelines-v2 — open from repo or set cwd to ml-pipelines-v2/")

ML_DIR = _find_ml_dir()
DATA_DIR = ML_DIR / "data"
MODEL_DIR = ML_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
BUILDER = ML_DIR / "build_master_datasets.py"
if BUILDER.exists() and (
    not (DATA_DIR / "donor_master.csv").exists()
    or not (DATA_DIR / "resident_master.csv").exists()
):
    subprocess.run([sys.executable, str(BUILDER)], check=True)

h = pd.read_csv(DATA_DIR / 'health_wellbeing_records.csv', parse_dates=['record_date'])
h = h.sort_values(['resident_id','record_date'])
h['score_lag1'] = h.groupby('resident_id')['general_health_score'].shift(1)
h['score_lag2'] = h.groupby('resident_id')['general_health_score'].shift(2)
h['bmi_lag1'] = h.groupby('resident_id')['bmi'].shift(1)
h['bmi_trend'] = h.groupby('resident_id')['bmi'].diff()
h['checkup_ok'] = h['medical_checkup_done'].astype(str).str.lower().eq('true').astype(int)
h = h.dropna(subset=['score_lag1','general_health_score'])
print(h.shape); h.head()


In [ ]:
feat = ['score_lag1','score_lag2','bmi_lag1','bmi_trend','checkup_ok']
m = h.dropna(subset=feat)
X = m[feat]
y = m['general_health_score']


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
prep = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
pipe_lin = Pipeline([('prep', prep), ('reg', LinearRegression())])
pipe_rf = Pipeline([('prep', prep), ('reg', RandomForestRegressor(200, max_depth=4, random_state=42))])


## 3. Modeling & Feature Selection


In [ ]:
pipe_lin.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)
print('LR coef', pipe_lin.named_steps['reg'].coef_)


## 4. Evaluation & Interpretation


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
for name, p in [('LR', pipe_lin), ('RF', pipe_rf)]:
    pr = p.predict(X_test)
    print(name, 'MAE', mean_absolute_error(y_test, pr), 'RMSE', mean_squared_error(y_test, pr)**0.5, 'R2', r2_score(y_test, pr))
res = y_test - pipe_rf.predict(X_test)
plt.scatter(pipe_rf.predict(X_test), res); plt.xlabel('pred'); plt.ylabel('resid'); plt.show()


## 5. Causal / Relationship Analysis

Use **statsmodels OLS** on train with HC3 — **lagged scores dominate** mechanically; causal claims about checkups require stronger designs.


In [ ]:
import statsmodels.api as sm
Xt = sm.add_constant(X_train)
print(sm.OLS(y_train, Xt).fit(cov_type='HC3').summary())


## 6. Deployment Notes

**Lighter pipeline — no .sav.** Would deploy the validated `sklearn` `Pipeline` via joblib behind a background worker; surfaces on **resident health** timeline view.
